# ⚡ ColBERT — Late Interaction Retrieval

## The Problem With Single-Vector Embeddings

Normal embedding search compresses an entire document into **one vector**:

```
"The cat sat on the red mat near the window"
                    ↓ encode
              [0.2, -0.5, 0.8, ...]   ← one vector for the whole sentence
```

A lot of detail is lost in that compression.

## The ColBERT Idea — One Vector Per Token

Instead of one vector per document, keep **one vector per word**:

```
"The cat sat on the red mat"
  ↓    ↓   ↓   ↓   ↓   ↓   ↓
 v1   v2  v3  v4  v5  v6  v7   ← 7 vectors!
```

At query time, each query token finds its **best matching** document token:

```
Query: "feline resting surface"
  "feline"  → best match in doc: "cat"   (score: 0.89)
  "resting" → best match in doc: "sat"   (score: 0.82)
  "surface" → best match in doc: "mat"   (score: 0.76)
  Final score = sum of best matches = 2.47
```

This is called **MaxSim** (maximum similarity) scoring.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Ready!")

Ready!


In [2]:
# ColBERT-style: embed each WORD separately

def encode_tokens(text):
    """Embed each word in the text separately."""
    words = text.lower().split()
    embeddings = model.encode(words)  # shape: (num_words, 384)
    return words, embeddings


# Example
doc = "The cat sat on the red mat near the window"
words, token_embs = encode_tokens(doc)

print(f"Document: '{doc}'")
print(f"Tokens:    {words}")
print(f"Shape:     {token_embs.shape}  ← {len(words)} words × 384 dims each")

Document: 'The cat sat on the red mat near the window'
Tokens:    ['the', 'cat', 'sat', 'on', 'the', 'red', 'mat', 'near', 'the', 'window']
Shape:     (10, 384)  ← 10 words × 384 dims each


In [3]:
# MaxSim scoring — the heart of ColBERT

def maxsim_score(query_text, doc_text):
    """
    ColBERT-style score:
    For each query token, find the best matching doc token.
    Sum those best scores.
    """
    q_words, q_embs = encode_tokens(query_text)
    d_words, d_embs = encode_tokens(doc_text)

    # For each query token, find max similarity to any doc token
    sim_matrix = cosine_similarity(q_embs, d_embs)  # shape: (n_query, n_doc)

    # MaxSim: for each query token, pick the highest doc token score
    max_per_query = sim_matrix.max(axis=1)  # shape: (n_query,)
    total_score   = float(max_per_query.sum())

    return total_score, sim_matrix, q_words, d_words


query = "feline resting surface"
doc   = "the cat sat on the mat"

score, sim_matrix, q_words, d_words = maxsim_score(query, doc)

print(f"Query: '{query}'")
print(f"Doc:   '{doc}'")
print(f"\nToken-level similarity matrix:")
print(f"{'':12}", end="")
for w in d_words:
    print(f"{w:8}", end="")
print()
for i, qw in enumerate(q_words):
    print(f"{qw:12}", end="")
    for j in range(len(d_words)):
        marker = " *" if sim_matrix[i].argmax() == j else "  "
        print(f"{sim_matrix[i,j]:6.3f}{marker}", end="")
    print()
print(f"\nMaxSim total score: {score:.3f}  (* = best match per query token)")

Query: 'feline resting surface'
Doc:   'the cat sat on the mat'

Token-level similarity matrix:
            the     cat     sat     on      the     mat     
feline       0.287   0.698 * 0.140   0.265   0.287   0.268  
resting      0.229   0.261   0.352 * 0.259   0.229   0.225  
surface      0.228   0.241   0.188   0.295   0.228   0.379 *

MaxSim total score: 1.429  (* = best match per query token)


In [4]:
# Compare: single-vector vs ColBERT-style on a tricky query

documents = [
    "The cat sat on the red mat near the window.",
    "Dogs love to run in the park and play fetch.",
    "A feline rested comfortably on a flat woven surface.",  # paraphrase of doc 1
    "The weather today is sunny with mild temperatures.",
]

query = "feline sitting on a surface"

# Single-vector search
q_emb     = model.encode(query)
doc_embs  = model.encode(documents)
sv_scores = cosine_similarity([q_emb], doc_embs)[0]

# ColBERT MaxSim scores
ms_scores = [maxsim_score(query, d)[0] for d in documents]

print(f"Query: '{query}'\n")
print(f"{'Document':<50} {'Single-Vec':>12} {'ColBERT':>10}")
print("-" * 75)
for i, doc in enumerate(documents):
    print(f"{doc[:48]:<50} {sv_scores[i]:>12.3f} {ms_scores[i]:>10.3f}")

print("\n💡 ColBERT can better match 'feline/cat' and 'surface/mat' through token-level alignment.")

Query: 'feline sitting on a surface'

Document                                             Single-Vec    ColBERT
---------------------------------------------------------------------------
The cat sat on the red mat near the window.               0.453      3.178
Dogs love to run in the park and play fetch.              0.309      2.263
A feline rested comfortably on a flat woven surf          0.770      4.374
The weather today is sunny with mild temperature          0.066      1.922

💡 ColBERT can better match 'feline/cat' and 'surface/mat' through token-level alignment.


## When to Use ColBERT

| | Single-Vector | ColBERT |
|---|---|---|
| **Speed** | Very fast | Slower (more vectors) |
| **Storage** | 1 vector/doc | N vectors/doc |
| **Accuracy** | Good | Better for paraphrases |
| **Best for** | General use | High-precision search |

**Production ColBERT:** use the `ragatouille` library which wraps ColBERTv2:
```python
from ragatouille import RAGPretrainedModel
rag = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")
rag.index(documents)
results = rag.search("your query")
```

**Paper:** ColBERT: Efficient and Effective Passage Search via Contextualized Late Interaction (Khattab & Zaharia, 2020)